## Aplicación de la arquitectura que ganó
En este notebook se usará la arquitectura ganadora. Se necesita cargar el modelo con los pesos que se guardaron durante la mejor época de entrenamiento, llamado "mejor_modelo_absoluto.pth". Para pasarlo a un transformer, se explicará como integrarlo al final del notebook.

## I: Preparación

### Importar librerías necesarias

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

## II: Bloques de la arquitectura ganadora

### Bloque backbone con EfficientNet-B3 

In [ ]:
class BackboneEfficientNetB3(nn.Module):
    def __init__(self):
        super().__init__()
        modelo_base = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        self.features = modelo_base.features
        self.indices_extraccion = [3, 5, 7]

    def forward(self, x):
        mapas_multiescala = []
        for i, capa in enumerate(self.features):
            x = capa(x)
            if i in self.indices_extraccion:
                mapas_multiescala.append(x)
            if i == max(self.indices_extraccion):
                break
        return mapas_multiescala

### Bloque de atención CBAM

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return self.sigmoid(out)

class CBAM(nn.Module):
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.channel_attention(x)
        out = out * self.spatial_attention(out)
        return out

### Bloque de fusión multiescala

In [ ]:
class MultiScaleFusionBlock(nn.Module):
    def __init__(self, in_channels_list, out_channels=384, target_size=(19, 19)):
        super(MultiScaleFusionBlock, self).__init__()
        self.projections = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1) 
            for in_channels in in_channels_list
        ])
        self.target_size = target_size

    def forward(self, mapas_cbam):
        fused_map = 0
        for i, mapa in enumerate(mapas_cbam):
            mapa_proyectado = self.projections[i](mapa)
            mapa_redimensionado = F.interpolate(
                mapa_proyectado, size=self.target_size, mode='bilinear', align_corners=False
            )
            fused_map = fused_map + mapa_redimensionado
        return fused_map

## III: Modulo exportador de la arquitectura ganadora

In [ ]:
class ExtractorLungX(nn.Module):
    """
    Este módulo contiene la arquitectura ganadora Híbrida (Con CBAM).
    Procesa las radiografías y devuelve un tensor espacial multiescala
    de dimensiones [Batch, 384, 19, 19] listo para ser ingerido por
    el Vision Transformer (DeiT) del equipo.
    """
    def __init__(self):
        super(ExtractorLungX, self).__init__()
        
        # Arquitectura base + Atenciones
        self.backbone = BackboneEfficientNetB3()
        canales = [48, 136, 384]
        
        # Módulos CBAM requeridos según los resultados de la investigación
        self.cbam_modules = nn.ModuleList([CBAM(c) for c in canales])
        self.fusion = MultiScaleFusionBlock(canales)

    def forward(self, x):
        mapas = self.backbone(x)
        
        mapas_listos = [cbam(mapa) for cbam, mapa in zip(self.cbam_modules, mapas)]
        
        mapa_fusionado = self.fusion(mapas_listos)

        # Salida, debería salir un tensor de tamaño [Batch, 384, 19, 19]
        return mapa_fusionado

## IV: Modo de uso

### Paso 1: Instanciar el modelo
```python
extractor = ExtractorLungX()
```
### Paso 2: Cargar los pesos del modelo entrenado
```python
extractor.load_state_dict(torch.load("mejor_modelo_absoluto.pth"), strict=False)
```  
Se usa `strict=False` porque el modelo tiene un módulo adicional que no se encuentra en el modelo entrenado.
### Paso 3: Conectar el transformer al extractor
```python
features = extractor(imagenes_batch)
tokens = patch_embedding(features)
```